# MinIO Basics with boto3
Connect to MinIO, manage buckets and objects.

In [ ]:
import boto3
import os
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url=os.getenv('MINIO_ENDPOINT', 'http://localhost:9000'),
    aws_access_key_id=os.getenv('MINIO_ACCESS_KEY', 'minioadmin'),
    aws_secret_access_key=os.getenv('MINIO_SECRET_KEY', 'minioadmin'),
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)
print('S3 client created:', s3.meta.endpoint_url)

In [ ]:
response = s3.list_buckets()
buckets = [b['Name'] for b in response['Buckets']]
print('Buckets:', buckets)

In [ ]:
bucket_name = 'notebook-test'
existing = [b['Name'] for b in s3.list_buckets()['Buckets']]
if bucket_name not in existing:
    s3.create_bucket(Bucket=bucket_name)
    print(f'Created bucket: {bucket_name}')
else:
    print(f'Bucket already exists: {bucket_name}')

In [ ]:
object_key = 'hello.txt'
content = 'Hello from JupyterLab and MinIO!'
s3.put_object(Bucket=bucket_name, Key=object_key, Body=content.encode('utf-8'))
print(f'Uploaded object: s3://{bucket_name}/{object_key}')

In [ ]:
response = s3.get_object(Bucket=bucket_name, Key=object_key)
downloaded = response['Body'].read().decode('utf-8')
print('Downloaded content:', downloaded)

In [ ]:
url = s3.generate_presigned_url(
    'get_object',
    Params={'Bucket': bucket_name, 'Key': object_key},
    ExpiresIn=3600
)
print('Presigned URL (valid 1 hour):')
print(url)

In [ ]:
s3.delete_object(Bucket=bucket_name, Key=object_key)
s3.delete_bucket(Bucket=bucket_name)
print(f'Cleaned up: deleted {object_key} and bucket {bucket_name}')